***
# Merge the arrival dataset

Create the merged arrival dataset for the selected destination and year. BTS supplies the flight records and scheduled departure timestamp. ASPM supplies planned demand at each flight's origin for the previous, current, and next clock hours. NOAA supplies the most recent origin-airport weather report available at or before scheduled departure.

The next-hour ASPM values are scheduled counts known ahead of time, not future operating results. The ASPM and NOAA airport keys and matched timestamps remain clearly named so their timing and airport alignment can be audited.
***

In [1]:
YEAR = 2024

AIRPORT = "JFK"

NOAA_TOLERANCE = "90min"

***
## Configure source and output paths

The concatenated inputs use `_ALL_{AIRPORT}_{YEAR}.csv`. The output is written to `data/merged`. The project-root detection supports starting Jupyter from either the project root or the `notebooks` directory.
***

In [2]:
from pathlib import Path

import pandas as pd

if (Path.cwd() / "data").is_dir() and (Path.cwd() / "notebooks").is_dir():
    PROJECT_ROOT = Path.cwd()
elif (Path.cwd().parent / "data").is_dir() and Path.cwd().name == "notebooks":
    PROJECT_ROOT = Path.cwd().parent
else:
    raise FileNotFoundError(
        "Start this notebook from the capstone project root or its notebooks directory"
    )

NOAA_TOLERANCE = pd.Timedelta(NOAA_TOLERANCE)

def data_paths(airport, year):
    return {
        "BTS": PROJECT_ROOT / f"data/bts/cleaned_{airport}_{year}.csv",
        "ASPM": PROJECT_ROOT / f"data/aspm/cleaned_{airport}_{year}.csv",
        "NOAA": PROJECT_ROOT / f"data/noaa/cleaned_{airport}_{year}.csv",
        "MERGED": PROJECT_ROOT / f"data/merged/{airport}_{year}_arrivals.csv",
    }

configured_files = pd.Series(
    {name: str(path) for name, path in data_paths(AIRPORT, YEAR).items()},
    name=f"{AIRPORT} {YEAR} files",
)
configured_files

BTS       /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
ASPM      /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
NOAA      /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
MERGED    /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
Name: JFK 2024 files, dtype: str

***
## Load and validate a year's cleaned sources

Parse all merge timestamps during loading. ASPM must have one row per airport-hour. Same-airport NOAA reports with identical timestamps are reported but retained; the stable timestamp sort makes the last same-time report in source order the match used by the backward as-of join.
***

In [3]:
def load_and_validate_sources(airport, year):
    paths = data_paths(airport, year)
    missing_files = [str(paths[name]) for name in ("BTS", "ASPM", "NOAA") if not paths[name].is_file()]
    if missing_files:
        raise FileNotFoundError(
            "Run the concatenation scripts first. Missing files: " + ", ".join(missing_files)
        )

    bts = pd.read_csv(paths["BTS"], parse_dates=["FlightDate", "DATE"])
    aspm = pd.read_csv(paths["ASPM"], parse_dates=["report_date", "DATE"])
    noaa = pd.read_csv(paths["NOAA"], parse_dates=["DATE"])

    aspm_columns = [
        "airport",
        "report_date",
        "Hour",
        "Scheduled Departures",
        "Scheduled Arrivals",
        "DATE",
    ]
    missing_aspm_columns = [column for column in aspm_columns if column not in aspm.columns]
    if missing_aspm_columns:
        raise KeyError(f"Missing required ASPM columns: {missing_aspm_columns}")
    aspm = aspm[aspm_columns].copy()

    required_columns = {
        "BTS": {"DATE", "FlightDate", "Origin", "Dest"},
        "ASPM": {"DATE", "airport", "report_date", "Hour"},
        "NOAA": {"DATE", "AIRPORT"},
    }
    source_frames = {"BTS": bts, "ASPM": aspm, "NOAA": noaa}
    missing_columns = {
        name: sorted(columns - set(source_frames[name].columns))
        for name, columns in required_columns.items()
        if columns - set(source_frames[name].columns)
    }
    if missing_columns:
        raise KeyError(f"Missing merge columns: {missing_columns}")

    invalid_dates = {
        "BTS missing DATE": int(bts["DATE"].isna().sum()),
        "ASPM missing DATE": int(aspm["DATE"].isna().sum()),
        "NOAA missing DATE": int(noaa["DATE"].isna().sum()),
    }
    if sum(invalid_dates.values()) > 0:
        raise ValueError(f"Invalid merge timestamps: {invalid_dates}")

    aspm_duplicate_keys = int(
        aspm.duplicated(subset=["airport", "DATE"], keep=False).sum()
    )
    if aspm_duplicate_keys > 0:
        raise ValueError(f"ASPM contains {aspm_duplicate_keys} duplicate airport-hour rows")

    source_validation = pd.Series({
        "BTS rows": len(bts),
        "ASPM rows": len(aspm),
        "NOAA rows": len(noaa),
        "ASPM duplicate airport-hour rows": aspm_duplicate_keys,
        "NOAA rows with duplicate airport-timestamps": int(
            noaa.duplicated(subset=["AIRPORT", "DATE"], keep=False).sum()
        ),
    }, name=f"{airport} {year} source validation")

    return paths, bts, aspm, noaa, source_validation

***
## Prepare destination arrivals and source keys

Keep flights arriving at the selected destination. ASPM and NOAA describe conditions at each flight's origin, so both joins use `Origin` as the airport key. Source timestamps and airport keys are renamed to remain explicit in the merged output.
***

In [4]:
def prepare_arrival_sources(bts, aspm, noaa, airport):
    arrivals = bts.loc[bts["Dest"] == airport].copy()
    arrivals = arrivals.sort_values("DATE", kind="stable").reset_index(drop=True)

    aspm = aspm.rename(columns={
        "airport": "AIRPORT",
        "report_date": "REPORT_DATE",
        "Hour": "HOUR",
        "Scheduled Departures": "SCHEDULED_DEPARTURES",
        "Scheduled Arrivals": "SCHEDULED_ARRIVALS",
    })

    noaa = noaa.rename(columns={"AIRPORT": "NOAA_AIRPORT", "DATE": "NOAA_DATE"})
    noaa = noaa.sort_values(["NOAA_DATE", "NOAA_AIRPORT"], kind="stable").reset_index(drop=True)

    return arrivals, aspm, noaa

***
## Join the previous, current, and next ASPM hours

For each scheduled departure, attach the origin airport's three ASPM schedule records based on the departure's local clock hour. Only planned arrival and departure counts are included. Each period receives its own airport, lookup timestamp, matched timestamp, and offset from scheduled departure.
***

In [5]:
ASPM_PERIOD_OFFSETS = {
    "PREVIOUS": -1,
    "CURRENT": 0,
    "NEXT": 1,
}

def attach_aspm(arrivals, aspm):
    arrivals = arrivals.copy()
    scheduled_departure_hour = arrivals["DATE"].dt.floor("h")

    for period, hour_offset in ASPM_PERIOD_OFFSETS.items():
        arrivals[f"ASPM_{period}_LOOKUP_DATE"] = (
            scheduled_departure_hour + pd.Timedelta(hours=hour_offset)
        )

    merged = arrivals
    for period in ASPM_PERIOD_OFFSETS:
        period_aspm = aspm.rename(columns={
            column: f"ASPM_{period}_{column}" for column in aspm.columns
        })
        merged = merged.merge(
            period_aspm,
            left_on=["Origin", f"ASPM_{period}_LOOKUP_DATE"],
            right_on=[f"ASPM_{period}_AIRPORT", f"ASPM_{period}_DATE"],
            how="left",
            validate="many_to_one",
        )
        merged[f"ASPM_{period}_OFFSET_MINUTES"] = (
            merged[f"ASPM_{period}_DATE"] - merged["DATE"]
        ).dt.total_seconds().div(60)

    return merged

***
## Join the latest available origin NOAA report

Use an airport-aware backward as-of join to select the most recent NOAA observation at the flight's origin at or before scheduled departure. The tolerance prevents a report more than 90 minutes old from being carried forward. No future or different-airport weather observation is allowed.
***

In [6]:
def attach_noaa(merged, noaa, tolerance):
    merged = merged.sort_values(["DATE", "Origin"], kind="stable").reset_index(drop=True)

    merged = pd.merge_asof(
        merged,
        noaa,
        left_on="DATE",
        right_on="NOAA_DATE",
        left_by="Origin",
        right_by="NOAA_AIRPORT",
        direction="backward",
        tolerance=tolerance,
        allow_exact_matches=True,
    )

    merged["NOAA_AGE_MINUTES"] = (
        merged["DATE"] - merged["NOAA_DATE"]
    ).dt.total_seconds().div(60)

    return merged

***
## Validate and save the merged arrival dataset

Confirm that the joins preserve the number of selected-destination arrival rows, ASPM and NOAA matches belong to the flight origin, all ASPM records match their requested clock hours, and NOAA never comes from the future. Missing source matches are reported rather than silently removed because source coverage and year boundaries can legitimately create gaps.
***

In [7]:
EXPECTED_ASPM_OFFSET_RANGES = {
    "PREVIOUS": (-119, -60),
    "CURRENT": (-59, 0),
    "NEXT": (1, 60),
}

def validate_merged_arrivals(arrivals, merged, tolerance):
    values = {
        "BTS arrival rows": len(arrivals),
        "merged rows": len(merged),
        "row-count difference": len(merged) - len(arrivals),
    }

    for period in ASPM_PERIOD_OFFSETS:
        matched_airport = merged[f"ASPM_{period}_AIRPORT"]
        matched_date = merged[f"ASPM_{period}_DATE"]
        lookup_date = merged[f"ASPM_{period}_LOOKUP_DATE"]
        offset_minutes = merged[f"ASPM_{period}_OFFSET_MINUTES"]
        minimum_offset, maximum_offset = EXPECTED_ASPM_OFFSET_RANGES[period]
        label = period.lower()

        values[f"missing ASPM {label} matches"] = int(matched_date.isna().sum())
        values[f"ASPM {label} airport mismatches"] = int((
            matched_airport.notna() & (matched_airport != merged["Origin"])
        ).sum())
        values[f"ASPM {label} timestamp mismatches"] = int((
            matched_date.notna() & (matched_date != lookup_date)
        ).sum())
        values[f"ASPM {label} offsets outside expected range"] = int((
            offset_minutes.notna()
            & ~offset_minutes.between(minimum_offset, maximum_offset)
        ).sum())

    values.update({
        "missing NOAA matches": int(merged["NOAA_DATE"].isna().sum()),
        "NOAA airport mismatches": int((
            merged["NOAA_AIRPORT"].notna()
            & (merged["NOAA_AIRPORT"] != merged["Origin"])
        ).sum()),
        "future NOAA matches": int((merged["NOAA_DATE"] > merged["DATE"]).sum()),
        "NOAA matches older than tolerance": int((
            merged["NOAA_AGE_MINUTES"] > tolerance.total_seconds() / 60
        ).sum()),
        "rows with duplicated BTS flight keys": int(merged.duplicated(
            subset=[
                "FlightDate",
                "Reporting_Airline",
                "Flight_Number_Reporting_Airline",
                "Origin",
                "Dest",
                "CRSDepTime",
            ],
            keep=False,
        ).sum()),
    })

    validation = pd.Series(values, name="merged-data validation")
    hard_failure_labels = [
        label for label in validation.index
        if label == "row-count difference"
        or "mismatches" in label
        or "outside expected range" in label
        or label == "future NOAA matches"
        or label == "NOAA matches older than tolerance"
    ]
    failures = validation.loc[hard_failure_labels]
    if (failures != 0).any():
        raise ValueError(f"Invalid merged arrival data:\n{failures[failures != 0]}")

    return validation

In [8]:
def merge_arrival_year(airport, year, tolerance):
    paths, bts, aspm, noaa, source_validation = load_and_validate_sources(airport, year)
    arrivals, aspm, noaa = prepare_arrival_sources(bts, aspm, noaa, airport)
    merged = attach_aspm(arrivals, aspm)
    merged = attach_noaa(merged, noaa, tolerance)
    merge_validation = validate_merged_arrivals(arrivals, merged, tolerance)

    paths["MERGED"].parent.mkdir(parents=True, exist_ok=True)
    merged.to_csv(paths["MERGED"], index=False)

    summary = {
        "year": year,
        "destination": airport,
        "rows": len(merged),
        "columns": len(merged.columns),
        "missing ASPM previous": int(merge_validation["missing ASPM previous matches"]),
        "missing ASPM current": int(merge_validation["missing ASPM current matches"]),
        "missing ASPM next": int(merge_validation["missing ASPM next matches"]),
        "missing NOAA": int(merge_validation["missing NOAA matches"]),
        "output": str(paths["MERGED"]),
    }

    return summary, source_validation, merge_validation

***
## Build the configured destination and year

Papermill replaces `YEAR` and `AIRPORT` in the tagged parameters cell before executing this single merge.
***

In [9]:
print(f"Merging {AIRPORT} arrivals for {YEAR}...")
summary, source_validation, merge_validation = merge_arrival_year(
    AIRPORT, YEAR, NOAA_TOLERANCE
)
print(f"Saved {summary['rows']:,} rows to {summary['output']}")

pd.Series(summary, name="merge summary")

Merging JFK arrivals for 2024...
Saved 104,555 rows to /Users/johnkyte/Projects/berkeley_ml_and_ai/capstone/data/merged/JFK_2024_arrivals.csv


year                                                                  2024
destination                                                            JFK
rows                                                                104555
columns                                                                 72
missing ASPM previous                                                    0
missing ASPM current                                                     0
missing ASPM next                                                        7
missing NOAA                                                           174
output                   /Users/johnkyte/Projects/berkeley_ml_and_ai/ca...
Name: merge summary, dtype: object

In [10]:
source_validation

BTS rows                                       104555
ASPM rows                                      439200
NOAA rows                                      634513
ASPM duplicate airport-hour rows                    0
NOAA rows with duplicate airport-timestamps         0
Name: JFK 2024 source validation, dtype: int64

In [11]:
merge_validation

BTS arrival rows                                104555
merged rows                                     104555
row-count difference                                 0
missing ASPM previous matches                        0
ASPM previous airport mismatches                     0
ASPM previous timestamp mismatches                   0
ASPM previous offsets outside expected range         0
missing ASPM current matches                         0
ASPM current airport mismatches                      0
ASPM current timestamp mismatches                    0
ASPM current offsets outside expected range          0
missing ASPM next matches                            7
ASPM next airport mismatches                         0
ASPM next timestamp mismatches                       0
ASPM next offsets outside expected range             0
missing NOAA matches                               174
NOAA airport mismatches                              0
future NOAA matches                                  0
NOAA match